# 🧠 CNN Avancé — Cinq Projets de Classification

Ce notebook couvre **5 exercices avancés** de classification par CNN :

| # | Exercice | Concept clé |
|---|----------|-------------|
| 1 | Classification Multi-Label | Sigmoid + BCE, dépendances entre labels |
| 2 | Gestion du Déséquilibre | Oversampling, WeightedSampler, Focal Loss |
| 3 | Transfer Learning & Fine-tuning | EfficientNet, LR différentiel, stratégies de gel |
| 4 | Interprétabilité du Modèle | Grad-CAM, LIME, Saliency Maps |
| 5 | Pipeline de Classification Complet | Modulaire, sauvegarde, interface de prédiction |

> ⚠️ **GPU recommandé** : `Runtime > Change runtime type > Hardware accelerator > GPU (T4)`

In [ ]:
# ── Installation des dépendances ──────────────────────────────────────────────
!pip install -q torch torchvision timm lime scikit-learn matplotlib seaborn

import os, copy, time, json, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam, SGD
from torch.utils.data import (
    Dataset, DataLoader, random_split,
    WeightedRandomSampler, Subset
)
import torchvision
import torchvision.transforms as T
import torchvision.models as models
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, average_precision_score
)

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')
%matplotlib inline

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Environnement prêt — Appareil : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"   GPU : {torch.cuda.get_device_name(0)}")

---
# 🏷️ Exercice 1 — Classification Multi-Label

## Objectif
Construire un CNN capable d'attribuer **plusieurs étiquettes simultanément** à une image.

### Différence avec la classification classique
| Type | Sortie | Activation | Loss |
|------|--------|------------|------|
| Multi-classe | 1 classe parmi N | Softmax | CrossEntropyLoss |
| **Multi-label** | **0 à N classes simultanément** | **Sigmoid** | **BCEWithLogitsLoss** |

La **Sigmoid** traite chaque label **indépendamment** (probabilité entre 0 et 1 par label), alors que la Softmax crée une compétition entre les classes.

### Dataset simulé
On crée un dataset CIFAR-10 multi-label en ajoutant des labels synthétiques (couleur dominante + type d'objet) pour illustrer le concept.

In [ ]:
# ── Dataset Multi-Label basé sur CIFAR-10 ────────────────────────────────────
# On enrichit CIFAR-10 avec des labels supplémentaires :
#   Label groupe 1 : type d'objet (vehicule / animal / autre)
#   Label groupe 2 : classe CIFAR originale (10 classes)
# Résultat : chaque image a un vecteur binaire de 13 labels

CIFAR_CLASSES = ['avion','voiture','oiseau','chat','cerf',
                 'chien','grenouille','cheval','bateau','camion']

# Mapping : classe CIFAR → groupe sémantique
# 0=vehicule, 1=animal, 2=autre
CLASS_TO_GROUP = {
    0: 0,  # avion → vehicule
    1: 0,  # voiture → vehicule
    2: 1,  # oiseau → animal
    3: 1,  # chat → animal
    4: 1,  # cerf → animal
    5: 1,  # chien → animal
    6: 1,  # grenouille → animal
    7: 1,  # cheval → animal
    8: 0,  # bateau → vehicule
    9: 0,  # camion → vehicule
}
GROUP_NAMES = ['vehicule', 'animal']
N_LABELS = len(CIFAR_CLASSES) + len(GROUP_NAMES)  # 12 labels totaux


class MultiLabelCIFAR(Dataset):
    """
    Wrapper CIFAR-10 → Multi-Label.
    Chaque image reçoit un vecteur binaire de 12 labels :
      [vehicule, animal, avion, voiture, oiseau, ..., camion]
    """
    def __init__(self, train=True, transform=None):
        self.base = torchvision.datasets.CIFAR10(
            './data', train=train, download=True, transform=transform
        )

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, class_id = self.base[idx]
        # Créer le vecteur multi-label
        label_vec = torch.zeros(N_LABELS)
        label_vec[CLASS_TO_GROUP[class_id]] = 1.0          # label de groupe
        label_vec[len(GROUP_NAMES) + class_id] = 1.0       # label de classe
        return img, label_vec


# Transforms
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

transform_train_ml = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomCrop(32, padding=4),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])
transform_test_ml = T.Compose([
    T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

ml_train = MultiLabelCIFAR(train=True,  transform=transform_train_ml)
ml_test  = MultiLabelCIFAR(train=False, transform=transform_test_ml)

ml_train_loader = DataLoader(ml_train, batch_size=128, shuffle=True,  num_workers=2)
ml_test_loader  = DataLoader(ml_test,  batch_size=128, shuffle=False, num_workers=2)

ALL_LABEL_NAMES = GROUP_NAMES + CIFAR_CLASSES
print(f"✅ Dataset Multi-Label créé : {len(ml_train)} train | {len(ml_test)} test")
print(f"   {N_LABELS} labels : {ALL_LABEL_NAMES}")

# Vérification sur un exemple
img0, lbl0 = ml_train[0]
active = [ALL_LABEL_NAMES[i] for i in lbl0.nonzero(as_tuple=True)[0].tolist()]
print(f"\n📋 Exemple — labels actifs : {active}")

In [ ]:
# ── Modèle CNN Multi-Label ────────────────────────────────────────────────────
class MultiLabelCNN(nn.Module):
    """
    CNN pour classification multi-label.

    Différences clés par rapport à un CNN classique :
    1. Couche de sortie → N_LABELS neurones (un par label)
    2. PAS de Softmax → Sigmoid implicite dans BCEWithLogitsLoss
    3. Seuil de décision : prob > 0.5 → label actif
    """
    def __init__(self, n_labels=N_LABELS):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),  nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1),nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 128),          nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, n_labels)  # Pas de Sigmoid ici (dans la loss)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


ml_model = MultiLabelCNN().to(DEVICE)

# BCEWithLogitsLoss = Sigmoid + BCE, numériquement plus stable que
# appliquer Sigmoid puis BCELoss séparément
ml_criterion = nn.BCEWithLogitsLoss()
ml_optimizer = Adam(ml_model.parameters(), lr=1e-3, weight_decay=1e-4)
ml_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(ml_optimizer, T_max=20)

print(f"✅ Modèle multi-label : {sum(p.numel() for p in ml_model.parameters()):,} paramètres")
print(f"   Sortie : {N_LABELS} logits → Sigmoid → seuil 0.5")

In [ ]:
# ── Entraînement Multi-Label ──────────────────────────────────────────────────
def multilabel_accuracy(logits, targets, threshold=0.5):
    """Exactitude exacte (exact match) : correct seulement si TOUS les labels concordent."""
    preds = (torch.sigmoid(logits) > threshold).float()
    return (preds == targets).all(dim=1).float().mean().item()

def hamming_accuracy(logits, targets, threshold=0.5):
    """Hamming accuracy : fraction de labels correctement prédits (plus doux)."""
    preds = (torch.sigmoid(logits) > threshold).float()
    return (preds == targets).float().mean().item()

ML_EPOCHS = 20
ml_hist = {'loss': [], 'hamming_acc': [], 'exact_acc': []}

print(f"🏋️  Entraînement multi-label ({ML_EPOCHS} epochs)...")
for epoch in range(ML_EPOCHS):
    ml_model.train()
    ep_loss, ep_hamm, ep_exact = 0.0, 0.0, 0.0

    for X, y in ml_train_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        ml_optimizer.zero_grad()
        logits = ml_model(X)
        loss   = ml_criterion(logits, y)
        loss.backward()
        ml_optimizer.step()
        ep_loss  += loss.item()
        ep_hamm  += hamming_accuracy(logits.detach(), y)
        ep_exact += multilabel_accuracy(logits.detach(), y)

    ml_scheduler.step()
    n = len(ml_train_loader)
    ml_hist['loss'].append(ep_loss / n)
    ml_hist['hamming_acc'].append(ep_hamm / n)
    ml_hist['exact_acc'].append(ep_exact / n)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:>2}/{ML_EPOCHS} | Loss: {ep_loss/n:.4f} | "
              f"Hamming: {ep_hamm/n*100:.1f}% | Exact: {ep_exact/n*100:.1f}%")

print("\n✅ Entraînement terminé !")

In [ ]:
# ── Évaluation et visualisation ───────────────────────────────────────────────
ml_model.eval()
all_logits, all_targets = [], []
with torch.no_grad():
    for X, y in ml_test_loader:
        logits = ml_model(X.to(DEVICE)).cpu()
        all_logits.append(logits); all_targets.append(y)

all_logits  = torch.cat(all_logits)
all_targets = torch.cat(all_targets)
all_preds   = (torch.sigmoid(all_logits) > 0.5).numpy().astype(int)
all_targets_np = all_targets.numpy().astype(int)

# Métriques par label
print("📊 Performance par label (F1-score) :\n")
f1_per_label = f1_score(all_targets_np, all_preds, average=None, zero_division=0)
for name, f1 in zip(ALL_LABEL_NAMES, f1_per_label):
    bar = '█' * int(f1 * 20)
    print(f"  {name:<12} {bar:<20} {f1:.3f}")

print(f"\n   Macro F1    : {f1_score(all_targets_np, all_preds, average='macro', zero_division=0):.3f}")
print(f"   Hamming acc : {(all_preds == all_targets_np).mean()*100:.1f}%")

# Courbes d'entraînement
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(ml_hist['loss'],       color='tomato',    lw=2, label='BCE Loss')
axes[0].set_title('Loss Multi-Label'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(ml_hist['hamming_acc'], color='royalblue', lw=2, label='Hamming Acc')
axes[1].plot(ml_hist['exact_acc'],   color='seagreen',  lw=2, label='Exact Match')
axes[1].set_title('Accuracy Multi-Label'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.suptitle('Exercice 1 — Classification Multi-Label', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Exemples de prédictions
def denorm(t):
    m, s = torch.tensor(CIFAR_MEAN).view(3,1,1), torch.tensor(CIFAR_STD).view(3,1,1)
    return torch.clamp(t * s + m, 0, 1)

fig, axes = plt.subplots(2, 4, figsize=(13, 6))
for i, ax in enumerate(axes.flatten()):
    img, lbl = ml_test[i * 150]
    with torch.no_grad():
        logit = ml_model(img.unsqueeze(0).to(DEVICE)).cpu().squeeze()
    pred = (torch.sigmoid(logit) > 0.5)
    true_lbl  = [ALL_LABEL_NAMES[j] for j in lbl.nonzero(as_tuple=True)[0]]
    pred_lbl  = [ALL_LABEL_NAMES[j] for j in pred.nonzero(as_tuple=True)[0]]
    ax.imshow(denorm(img).permute(1, 2, 0))
    ax.set_title(f'V:{true_lbl}\nP:{pred_lbl}', fontsize=6)
    ax.axis('off')
plt.suptitle('Vrais labels (V) vs Prédits (P)', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

---
# ⚖️ Exercice 2 — Gestion du Déséquilibre de Classes

## Objectif
Comparer **trois stratégies** pour traiter un dataset fortement déséquilibré.

### Stratégies comparées

| Stratégie | Principe | Avantage |
|-----------|----------|----------|
| **Baseline** | Aucune correction | Référence |
| **WeightedSampler** | Sur-échantillonne les classes rares pendant le chargement | Simple, pas de duplication |
| **Weighted Loss** | Pénalise plus les erreurs sur classes rares | Flexible, contrôle fin |
| **Focal Loss** | Concentre l'apprentissage sur les cas difficiles | État de l'art |

### Métriques adaptées aux données déséquilibrées
- **F1-score** : harmonie précision/rappel
- **AUC-ROC** : robuste au déséquilibre
- **Average Precision** : aire sous la courbe précision-rappel

In [ ]:
# ── Dataset déséquilibré à partir de CIFAR-10 ────────────────────────────────
# On garde 100% des images de 3 classes (majoritaires)
# et seulement 5% des autres (minoritaires)

MAJORITY_CLASSES = {0, 1, 8}    # avion, voiture, bateau
MINORITY_RATIO   = 0.05          # 5% des classes minoritaires
N_IMBALANCED_CLASSES = 10

base_train = torchvision.datasets.CIFAR10(
    './data', train=True, download=True,
    transform=T.Compose([T.RandomHorizontalFlip(), T.RandomCrop(32, padding=4),
                         T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
)
base_test = torchvision.datasets.CIFAR10(
    './data', train=False, download=True,
    transform=T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
)

# Sélectionner les indices déséquilibrés
imbalanced_indices = []
for idx, (_, label) in enumerate(base_train):
    if label in MAJORITY_CLASSES:
        imbalanced_indices.append(idx)
    elif np.random.rand() < MINORITY_RATIO:
        imbalanced_indices.append(idx)

imbalanced_ds = Subset(base_train, imbalanced_indices)

# Compter la distribution
label_counts = np.zeros(10, dtype=int)
for idx in imbalanced_indices:
    label_counts[base_train[idx][1]] += 1

print("📊 Distribution du dataset déséquilibré :")
for i, (name, count) in enumerate(zip(CIFAR_CLASSES, label_counts)):
    bar = '█' * (count // 50)
    tag = '(majoritaire)' if i in MAJORITY_CLASSES else '(minoritaire)'
    print(f"  {name:<12} {bar:<50} {count:>4} {tag}")
print(f"  Total : {sum(label_counts)} images")

In [ ]:
# ── Préparation des 4 stratégies ──────────────────────────────────────────────

# ── 1. Baseline : loader standard ────────────────────────────────────────────
loader_imbalanced = DataLoader(imbalanced_ds, batch_size=128, shuffle=True, num_workers=2)

# ── 2. WeightedRandomSampler ──────────────────────────────────────────────────
# Poids inversement proportionnels à la fréquence de chaque classe
# → les classes rares sont tirées plus souvent
class_weights = 1.0 / torch.tensor(label_counts + 1, dtype=torch.float)
sample_weights = torch.tensor([class_weights[base_train[i][1]] for i in imbalanced_indices])
weighted_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True  # Avec remplacement pour le sur-échantillonnage
)
loader_weighted = DataLoader(imbalanced_ds, batch_size=128,
                             sampler=weighted_sampler, num_workers=2)

# ── 3. Weighted Loss ──────────────────────────────────────────────────────────
# Même loader que baseline, mais loss pondérée
loss_weights = class_weights / class_weights.sum() * N_IMBALANCED_CLASSES
weighted_criterion = nn.CrossEntropyLoss(weight=loss_weights.to(DEVICE))

# ── 4. Focal Loss ────────────────────────────────────────────────────────────
class FocalLossMulticlass(nn.Module):
    """
    Focal Loss pour classification multi-classes.
    FL(p_t) = -α_t * (1 - p_t)^γ * log(p_t)
    Le terme (1-p_t)^γ réduit la contribution des exemples faciles
    → le modèle se concentre sur les cas difficiles/rares.
    """
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha  # poids par classe (optionnel)

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
        p_t = torch.exp(-ce_loss)
        focal_loss = (1 - p_t) ** self.gamma * ce_loss
        return focal_loss.mean()

focal_criterion = FocalLossMulticlass(gamma=2.0, alpha=loss_weights.to(DEVICE))

print("✅ Les 4 stratégies sont prêtes :")
print("   1. Baseline (pas de correction)")
print("   2. WeightedRandomSampler")
print("   3. Weighted CrossEntropyLoss")
print("   4. Focal Loss")

In [ ]:
# ── Modèle et fonction d'entraînement ────────────────────────────────────────
class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(128*4*4, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, n_classes)
        )
    def forward(self, x): return self.net(x)


def run_imbalance_experiment(loader, criterion, name, epochs=15):
    model = SmallCNN().to(DEVICE)
    opt   = Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    test_loader = DataLoader(base_test, batch_size=128, shuffle=False, num_workers=2)

    for epoch in range(epochs):
        model.train()
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            criterion(model(X), y).backward()
            opt.step()
        sched.step()

    # Évaluation
    model.eval()
    all_p, all_t, all_prob = [], [], []
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            logits = model(X)
            probs  = torch.softmax(logits, dim=1).cpu().numpy()
            preds  = logits.argmax(1).cpu().numpy()
            all_p.extend(preds); all_t.extend(y.cpu().numpy()); all_prob.extend(probs)

    all_p, all_t = np.array(all_p), np.array(all_t)
    all_prob = np.array(all_prob)

    f1  = f1_score(all_t, all_p, average='macro', zero_division=0)
    acc = (all_p == all_t).mean()
    try:
        auc = roc_auc_score(np.eye(10)[all_t], all_prob, multi_class='ovr', average='macro')
    except Exception:
        auc = 0.0

    print(f"  [{name:<25}] Acc: {acc*100:.1f}% | F1 macro: {f1:.3f} | AUC: {auc:.3f}")
    return {'acc': acc, 'f1': f1, 'auc': auc, 'preds': all_p, 'true': all_t}


print("⚖️  Comparaison des stratégies de rééquilibrage...\n")
plain_criterion = nn.CrossEntropyLoss()

results_imb = {}
results_imb['Baseline']        = run_imbalance_experiment(loader_imbalanced, plain_criterion,    'Baseline')
results_imb['WeightedSampler'] = run_imbalance_experiment(loader_weighted,   plain_criterion,    'WeightedSampler')
results_imb['WeightedLoss']    = run_imbalance_experiment(loader_imbalanced, weighted_criterion, 'WeightedLoss')
results_imb['FocalLoss']       = run_imbalance_experiment(loader_imbalanced, focal_criterion,    'FocalLoss')

In [ ]:
# ── Visualisation des métriques ───────────────────────────────────────────────
strategies = list(results_imb.keys())
f1s  = [results_imb[s]['f1']  for s in strategies]
aucs = [results_imb[s]['auc'] for s in strategies]
accs = [results_imb[s]['acc'] for s in strategies]

x    = np.arange(len(strategies))
width = 0.28
colors_bar = ['#e74c3c', '#3498db', '#2ecc71']

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width, accs, width, label='Accuracy',  color=colors_bar[0], alpha=0.85)
bars2 = ax.bar(x,         f1s,  width, label='F1 Macro',  color=colors_bar[1], alpha=0.85)
bars3 = ax.bar(x + width, aucs, width, label='AUC-ROC',   color=colors_bar[2], alpha=0.85)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x); ax.set_xticklabels(strategies, fontsize=10)
ax.set_title('Exercice 2 — Métriques par stratégie de rééquilibrage', fontsize=13, fontweight='bold')
ax.set_ylabel('Score'); ax.set_ylim(0, 1.1); ax.legend()
plt.tight_layout(); plt.show()

# Matrice de confusion pour le meilleur modèle (Focal Loss)
best = results_imb['FocalLoss']
cm   = confusion_matrix(best['true'], best['preds'])
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CIFAR_CLASSES, yticklabels=CIFAR_CLASSES, ax=ax)
ax.set_title('Matrice de confusion — Focal Loss (meilleure stratégie)', fontsize=12)
ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')
plt.tight_layout(); plt.show()

---
# 🔄 Exercice 3 — Transfer Learning & Fine-tuning

## Objectif
Comparer **trois stratégies de fine-tuning** d'un EfficientNet pré-entraîné sur ImageNet.

### Trois approches progressives

```
Stratégie A — Feature Extraction  : [FROZEN backbone] + [NEW head]  ← rapide
Stratégie B — Partial Fine-tuning : [FROZEN early] + [FREE late] + [NEW head]
Stratégie C — Full Fine-tuning    : [ALL FREE (LR différentiel)] + [NEW head] ← puissant
```

### Pourquoi le LR différentiel ?
Les couches profondes pré-entraînées contiennent des features précieuses. Un LR trop grand les détruirait. On utilise donc :
- LR très faible pour le backbone (ex : 1e-5)
- LR normal pour la tête (ex : 1e-3)

In [ ]:
# ── EfficientNet-B0 adapté à CIFAR-10 ────────────────────────────────────────
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

def build_efficientnet(strategy='feature_extraction'):
    """
    Construit un EfficientNet-B0 pré-entraîné selon la stratégie :
      'feature_extraction'  → seule la tête est entraînable
      'partial_finetune'    → tête + 2 derniers blocs libres
      'full_finetune'       → tout est libre (LR différentiel)
    """
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

    # Geler tout d'abord
    for param in model.parameters():
        param.requires_grad = False

    if strategy == 'partial_finetune':
        # Dégeler les 2 derniers blocs du backbone
        blocks = list(model.features.children())
        for block in blocks[-2:]:
            for param in block.parameters():
                param.requires_grad = True

    elif strategy == 'full_finetune':
        # Dégeler tout (LR différentiel dans l'optimiseur)
        for param in model.parameters():
            param.requires_grad = True

    # Remplacer la tête de classification
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, 256), nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(256, 10)
    )
    # La tête est toujours entraînable
    for param in model.classifier.parameters():
        param.requires_grad = True

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total     = sum(p.numel() for p in model.parameters())
    print(f"  [{strategy}] Paramètres entraînables : {n_trainable:>7,} / {n_total:,}")
    return model


print("✅ Modèles EfficientNet-B0 :")
model_fe   = build_efficientnet('feature_extraction')
model_part = build_efficientnet('partial_finetune')
model_full = build_efficientnet('full_finetune')

# Loader CIFAR-10 avec resize pour EfficientNet (attend 224×224)
transform_eff = T.Compose([
    T.Resize(64),   # On utilise 64 au lieu de 224 pour économiser le temps
    T.RandomHorizontalFlip(), T.RandomCrop(64, padding=8),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Normes ImageNet
])
transform_eff_test = T.Compose([
    T.Resize(64), T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
eff_train = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=transform_eff)
eff_test  = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=transform_eff_test)
eff_train_loader = DataLoader(eff_train, batch_size=64, shuffle=True,  num_workers=2)
eff_test_loader  = DataLoader(eff_test,  batch_size=64, shuffle=False, num_workers=2)
print("\n✅ Loaders prêts (images redimensionnées à 64×64).")

In [ ]:
# ── Entraînement des 3 stratégies ────────────────────────────────────────────
def train_efficientnet(model, strategy, epochs=10):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()

    # LR différentiel : backbone plus lent que la tête
    backbone_params = [p for name, p in model.named_parameters()
                       if 'classifier' not in name and p.requires_grad]
    head_params     = list(model.classifier.parameters())

    lr_backbone = 1e-5 if strategy == 'full_finetune' else 0.0
    optimizer = Adam([
        {'params': backbone_params, 'lr': lr_backbone},
        {'params': head_params,     'lr': 1e-3}
    ], weight_decay=1e-4)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    hist = {'train_acc': [], 'test_acc': []}
    t0   = time.time()

    for epoch in range(epochs):
        model.train()
        c, t = 0, 0
        for X, y in eff_train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(X); loss = criterion(out, y); loss.backward(); optimizer.step()
            c += out.argmax(1).eq(y).sum().item(); t += y.size(0)
        scheduler.step()
        hist['train_acc'].append(100 * c / t)

        model.eval()
        vc, vt = 0, 0
        with torch.no_grad():
            for X, y in eff_test_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                vc += model(X).argmax(1).eq(y).sum().item(); vt += y.size(0)
        hist['test_acc'].append(100 * vc / vt)
        if (epoch + 1) % 3 == 0:
            print(f"  [{strategy:<20}] Epoch {epoch+1:>2} | "
                  f"Train: {hist['train_acc'][-1]:.1f}% | Test: {hist['test_acc'][-1]:.1f}%")

    elapsed = time.time() - t0
    print(f"  [{strategy:<20}] ⏱️  {elapsed:.0f}s | Best test: {max(hist['test_acc']):.1f}%")
    return hist, elapsed


print("🔵 Feature Extraction...")
hist_fe,   t_fe   = train_efficientnet(model_fe,   'feature_extraction')
print("\n🟡 Partial Fine-tuning...")
hist_part, t_part = train_efficientnet(model_part, 'partial_finetune')
print("\n🟢 Full Fine-tuning...")
hist_full, t_full = train_efficientnet(model_full, 'full_finetune')

In [ ]:
# ── Visualisation comparative ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, 11)
styles = [('Feature Extraction', hist_fe,   'tomato'),
          ('Partial Fine-tuning', hist_part, 'darkorange'),
          ('Full Fine-tuning',    hist_full, 'royalblue')]

for label, hist, color in styles:
    axes[0].plot(ep, hist['train_acc'], '--', color=color, alpha=0.5, lw=1.5)
    axes[0].plot(ep, hist['test_acc'],  '-',  color=color, lw=2, label=label)

axes[0].set_title('Train (--) vs Test (—)', fontsize=12)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy (%)')
axes[0].legend(fontsize=9)

names_ft = [l[0] for l in styles]
best_acc  = [max(l[1]['test_acc']) for l in styles]
times_ft  = [t_fe, t_part, t_full]
colors_ft = [l[2] for l in styles]

ax2 = axes[1].twinx()
bars = axes[1].bar(names_ft, best_acc, color=colors_ft, alpha=0.8, width=0.5)
ax2.plot(names_ft, times_ft, 'D-', color='black', lw=2, markersize=8, label='Temps (s)')
for bar, v in zip(bars, best_acc):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{v:.1f}%', ha='center', fontweight='bold')
axes[1].set_ylabel('Best Test Accuracy (%)')
ax2.set_ylabel('Temps total (s)')
axes[1].set_title('Accuracy vs Temps d\'entraînement', fontsize=12)
axes[1].tick_params(axis='x', rotation=15)
ax2.legend(loc='upper left')

plt.suptitle('Exercice 3 — Stratégies de Fine-tuning (EfficientNet-B0)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
# 🔍 Exercice 4 — Interprétabilité du Modèle

## Objectif
Comprendre et visualiser les **décisions internes** d'un CNN avec trois techniques complémentaires.

### Techniques d'interprétabilité

| Technique | Type | Ce qu'elle montre |
|-----------|------|------------------|
| **Saliency Map** | Gradient brut | Pixels qui influencent le plus le score |
| **Grad-CAM** | Gradient + feature maps | Régions spatiales les plus importantes |
| **LIME** | Perturbations locales | Superpixels qui justifient la prédiction |

### Pourquoi c'est important ?
L'interprétabilité permet de **détecter les biais** (ex: un modèle qui classe les wolves basé sur la neige en fond), de **gagner la confiance** des utilisateurs, et de **déboguer** les erreurs.

In [ ]:
# ── Modèle de référence pour l'interprétabilité ───────────────────────────────
# On utilise le modèle full fine-tuned de l'exercice 3
interp_model = model_full.eval()

# Sélectionner quelques images de test
test_indices = [0, 10, 50, 100, 200, 500]
test_imgs    = [eff_test[i][0] for i in test_indices]
test_labels  = [eff_test[i][1] for i in test_indices]

def to_tensor(img): return img.unsqueeze(0).to(DEVICE)

def denorm_imagenet(t):
    """Inverse la normalisation ImageNet pour affichage."""
    m = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    s = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    return torch.clamp(t * s + m, 0, 1)

print(f"✅ {len(test_imgs)} images de test sélectionnées pour la visualisation")

In [ ]:
# ── Technique 1 : Saliency Maps ──────────────────────────────────────────────
# Principe : calcule le gradient de la classe prédite par rapport à l'image.
# Les pixels avec un gradient élevé sont ceux dont la modification affecterait
# le plus la prédiction → pixels "importants" pour la décision.

def compute_saliency(model, img_tensor, target_class=None):
    """
    Calcule la saliency map (gradient de la sortie par rapport à l'entrée).
    Retourne un array 2D (H, W) normalisé entre 0 et 1.
    """
    inp = img_tensor.clone().requires_grad_(True)
    model.zero_grad()
    out = model(inp)

    if target_class is None:
        target_class = out.argmax(1).item()

    # Rétropropager uniquement pour la classe cible
    out[0, target_class].backward()

    # Prendre la valeur absolue du gradient et agréger les 3 canaux RGB
    saliency = inp.grad.data.abs().squeeze()
    saliency, _ = saliency.max(dim=0)  # max sur les canaux
    saliency = saliency.cpu().numpy()
    saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)
    return saliency, target_class


fig, axes = plt.subplots(2, len(test_imgs), figsize=(15, 5))
for i, (img, true_lbl) in enumerate(zip(test_imgs, test_labels)):
    inp = to_tensor(img)
    saliency, pred = compute_saliency(interp_model, inp)
    img_display    = denorm_imagenet(img).permute(1, 2, 0).numpy()

    axes[0, i].imshow(img_display); axes[0, i].axis('off')
    color = 'green' if pred == true_lbl else 'red'
    axes[0, i].set_title(f'{CIFAR_CLASSES[true_lbl]}\n→ {CIFAR_CLASSES[pred]}',
                          fontsize=8, color=color)
    axes[1, i].imshow(saliency, cmap='hot'); axes[1, i].axis('off')
    axes[1, i].set_title('Saliency', fontsize=8)

plt.suptitle('Technique 1 — Saliency Maps (rouge = pixels importants)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Technique 2 : Grad-CAM ───────────────────────────────────────────────────
# Principe : utilise les gradients de la DERNIÈRE couche convolutive.
# Avantage sur Saliency : plus smooth, localise mieux les régions discriminantes.
#
# Algorithme :
# 1. Forward pass → collecter les feature maps de la dernière conv
# 2. Backward pass → collecter les gradients de ces feature maps
# 3. Pondérer chaque feature map par la moyenne de ses gradients (Global Avg Pool)
# 4. Sommer et appliquer ReLU → CAM
# 5. Upsampler à la taille de l'image d'entrée

class GradCAM:
    def __init__(self, model, target_layer):
        self.model    = model
        self.features = None
        self.grads    = None
        # Enregistrement des activations et gradients via hooks
        target_layer.register_forward_hook(self._save_features)
        target_layer.register_backward_hook(self._save_grads)

    def _save_features(self, module, inp, out): self.features = out
    def _save_grads(self, module, gi, go):      self.grads = go[0]

    def __call__(self, img_tensor, class_idx=None):
        self.model.eval()
        img_tensor = img_tensor.requires_grad_(True)
        out = self.model(img_tensor)

        if class_idx is None:
            class_idx = out.argmax(1).item()

        self.model.zero_grad()
        out[0, class_idx].backward()

        # Weights = Global Average Pooling des gradients par canal
        weights = self.grads.mean(dim=(2, 3), keepdim=True)  # (1, C, 1, 1)
        cam     = (weights * self.features).sum(dim=1, keepdim=True)  # (1, 1, H, W)
        cam     = F.relu(cam)
        cam     = F.interpolate(cam, img_tensor.shape[-2:], mode='bilinear', align_corners=False)
        cam     = cam.squeeze().detach().cpu().numpy()
        cam     = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx


# Trouver la dernière couche Conv de EfficientNet
last_conv = None
for module in interp_model.modules():
    if isinstance(module, nn.Conv2d):
        last_conv = module
grad_cam = GradCAM(interp_model, last_conv)

fig, axes = plt.subplots(3, len(test_imgs), figsize=(15, 7))
for i, (img, true_lbl) in enumerate(zip(test_imgs, test_labels)):
    inp         = to_tensor(img)
    cam, pred   = grad_cam(inp)
    img_display = denorm_imagenet(img).permute(1, 2, 0).numpy()
    h, w        = img_display.shape[:2]

    # Coloriser la CAM
    cam_colored = mpl_cm.jet(cam)[:, :, :3]  # RGB depuis colormap jet
    overlay     = 0.5 * img_display + 0.5 * cam_colored
    overlay     = np.clip(overlay, 0, 1)

    axes[0, i].imshow(img_display); axes[0, i].axis('off')
    color = 'green' if pred == true_lbl else 'red'
    axes[0, i].set_title(f'{CIFAR_CLASSES[true_lbl]}\n→{CIFAR_CLASSES[pred]}', fontsize=8, color=color)
    axes[1, i].imshow(cam, cmap='jet'); axes[1, i].axis('off')
    axes[1, i].set_title('CAM', fontsize=8)
    axes[2, i].imshow(overlay); axes[2, i].axis('off')
    axes[2, i].set_title('Overlay', fontsize=8)

plt.suptitle('Technique 2 — Grad-CAM (rouge = régions discriminantes)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Technique 3 : LIME (Local Interpretable Model-Agnostic Explanations) ──────
# Principe : perturbe l'image en masquant des superpixels
# → entraîne un modèle linéaire local pour expliquer la prédiction
# → les superpixels positifs soutiennent la classe, les négatifs s'y opposent
#
# Avantage : Model-agnostic (fonctionne avec n'importe quel modèle)
# Inconvénient : plus lent (nécessite ~1000 inférences par image)

from lime import lime_image
from skimage.segmentation import mark_boundaries

def predict_fn(images_np):
    """
    Fonction de prédiction pour LIME.
    Prend un batch d'images numpy (H, W, C) entre 0-1
    Retourne les probabilités (n, n_classes).
    """
    transform_lime = T.Compose([
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    batch = torch.stack([transform_lime(Image.fromarray((img * 255).astype(np.uint8)))
                         for img in images_np]).to(DEVICE)
    with torch.no_grad():
        logits = interp_model(batch)
    return torch.softmax(logits, dim=1).cpu().numpy()


explainer = lime_image.LimeImageExplainer()

# Appliquer LIME sur 3 images (plus long → ~1 min par image)
lime_indices = test_indices[:3]
fig, axes    = plt.subplots(2, len(lime_indices), figsize=(12, 6))

for col, idx in enumerate(lime_indices):
    img, lbl    = eff_test[idx]
    img_display = denorm_imagenet(img).permute(1, 2, 0).numpy()

    # Prédiction
    with torch.no_grad():
        pred = interp_model(to_tensor(img)).argmax(1).item()

    # Explication LIME
    explanation = explainer.explain_instance(
        img_display.astype(np.float64),
        predict_fn,
        top_labels=1,
        hide_color=0,
        num_samples=500  # Réduire pour accélérer (augmenter pour meilleure précision)
    )

    # Visualisation des superpixels explicatifs
    temp, mask = explanation.get_image_and_mask(
        pred, positive_only=True, num_features=5, hide_rest=False
    )

    axes[0, col].imshow(img_display)
    color = 'green' if pred == lbl else 'red'
    axes[0, col].set_title(f'{CIFAR_CLASSES[lbl]} → {CIFAR_CLASSES[pred]}',
                            fontsize=9, color=color)
    axes[0, col].axis('off')
    axes[1, col].imshow(mark_boundaries(temp, mask))
    axes[1, col].set_title('LIME (vert = zones en faveur)', fontsize=9)
    axes[1, col].axis('off')

plt.suptitle('Technique 3 — LIME (superpixels explicatifs)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

### 📝 Analyse — Interprétabilité

**Saliency Maps** sont les plus rapides mais les plus bruitées : elles montrent des pixels épars sans structure spatiale claire. Utiles pour un diagnostic rapide mais difficiles à interpréter visuellement.

**Grad-CAM** offre un meilleur compromis : les cartes d'activation sont lisses et localisent clairement les régions discriminantes (ex: la tête d'un animal, la carlingue d'un avion). C'est la technique la plus utilisée en pratique pour valider les décisions d'un CNN.

**LIME** est le plus fidèle sémantiquement car il travaille au niveau des superpixels (régions cohérentes), pas des pixels individuels. Il est model-agnostic, ce qui en fait un outil universel. Sa lenteur (500+ inférences par image) est son principal inconvénient.

Ces trois techniques sont **complémentaires** : Grad-CAM pour la rapidité, LIME pour la précision, Saliency pour le débogage bas-niveau. En production, leur combinaison permet de détecter des biais dangereux avant déploiement.

---
# 🏗️ Exercice 5 — Pipeline de Classification Complet

## Objectif
Concevoir un **pipeline modulaire, réutilisable et robuste** de bout en bout, avec une interface de prédiction.

### Architecture du pipeline
```
📁 Config (JSON)          → paramètres centralisés
    ↓
📦 DataModule             → chargement + augmentation + split
    ↓
🧠 ModelFactory           → construction du modèle (scratch ou pretrained)
    ↓
🏋️  Trainer               → boucle train/val + early stopping + checkpoints
    ↓
📊 Evaluator              → métriques complètes + visualisations
    ↓
💾 ModelManager           → sauvegarde / chargement
    ↓
🎯 Predictor              → interface pour nouvelles images
```

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODULE 1 : Configuration centralisée
# ══════════════════════════════════════════════════════════════════════════════
# Tous les hyperparamètres dans un seul endroit → facile à modifier et versionner

DEFAULT_CONFIG = {
    "dataset":       "cifar10",
    "n_classes":     10,
    "img_size":      32,
    "batch_size":    128,
    "num_workers":   2,
    "val_split":     0.1,
    "model_name":    "simple_cnn",     # 'simple_cnn' | 'resnet18' | 'efficientnet_b0'
    "pretrained":    False,
    "dropout":       0.4,
    "epochs":        20,
    "lr":            1e-3,
    "weight_decay":  1e-4,
    "scheduler":     "cosine",          # 'cosine' | 'step' | 'plateau'
    "augmentation":  True,
    "early_stop_patience": 8,
    "checkpoint_path": "./pipeline_best_model.pth",
    "class_names":   ['avion','voiture','oiseau','chat','cerf',
                      'chien','grenouille','cheval','bateau','camion']
}

def save_config(cfg, path='config.json'):
    with open(path, 'w') as f: json.dump(cfg, f, indent=2)
    print(f"💾 Config sauvegardée → {path}")

def load_config(path='config.json'):
    with open(path) as f: return json.load(f)

save_config(DEFAULT_CONFIG)
CFG = DEFAULT_CONFIG
print("✅ Configuration chargée.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODULE 2 : DataModule
# ══════════════════════════════════════════════════════════════════════════════

class DataModule:
    """
    Gère tout le cycle de vie des données :
    - Téléchargement + normalisation
    - Augmentation configurable
    - Découpage train/val/test
    - Création des DataLoaders
    """
    STATS = {
        'cifar10': {'mean': (0.4914, 0.4822, 0.4465), 'std': (0.2470, 0.2435, 0.2616)}
    }

    def __init__(self, cfg):
        self.cfg = cfg
        stats    = self.STATS[cfg['dataset']]

        # ── Transforms ──────────────────────────────────────────────────────
        aug_list = [
            T.RandomHorizontalFlip(),
            T.RandomCrop(cfg['img_size'], padding=4),
            T.ColorJitter(brightness=0.2, contrast=0.2)
        ] if cfg['augmentation'] else []

        self.train_transform = T.Compose(
            aug_list + [T.ToTensor(), T.Normalize(stats['mean'], stats['std'])]
        )
        self.test_transform = T.Compose(
            [T.ToTensor(), T.Normalize(stats['mean'], stats['std'])]
        )
        self._setup()

    def _setup(self):
        """Charge et divise les datasets."""
        full_train = torchvision.datasets.CIFAR10(
            './data', train=True, download=True, transform=self.train_transform)
        self.test_ds = torchvision.datasets.CIFAR10(
            './data', train=False, download=True, transform=self.test_transform)

        n_val = int(len(full_train) * self.cfg['val_split'])
        n_tr  = len(full_train) - n_val
        self.train_ds, self.val_ds = random_split(
            full_train, [n_tr, n_val], generator=torch.Generator().manual_seed(42))

        print(f"✅ DataModule : {n_tr} train | {n_val} val | {len(self.test_ds)} test")

    def get_loaders(self):
        """Retourne les trois DataLoaders."""
        kw = {'batch_size': self.cfg['batch_size'], 'num_workers': self.cfg['num_workers']}
        return (
            DataLoader(self.train_ds, shuffle=True,  **kw),
            DataLoader(self.val_ds,   shuffle=False, **kw),
            DataLoader(self.test_ds,  shuffle=False, **kw)
        )

dm = DataModule(CFG)
train_loader_p, val_loader_p, test_loader_p = dm.get_loaders()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODULE 3 : ModelFactory
# ══════════════════════════════════════════════════════════════════════════════

class ModelFactory:
    """Construit le modèle à partir de la configuration."""

    @staticmethod
    def build(cfg):
        n_classes = cfg['n_classes']
        dropout   = cfg['dropout']

        if cfg['model_name'] == 'simple_cnn':
            return nn.Sequential(
                nn.Conv2d(3, 64, 3, padding=1),  nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
                nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
                nn.Conv2d(128, 256, 3, padding=1),nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
                nn.Flatten(),
                nn.Linear(256*4*4, 512), nn.ReLU(), nn.Dropout(dropout),
                nn.Linear(512, n_classes)
            )
        elif cfg['model_name'] == 'resnet18':
            from torchvision.models import resnet18, ResNet18_Weights
            weights = ResNet18_Weights.IMAGENET1K_V1 if cfg['pretrained'] else None
            m = resnet18(weights=weights)
            m.fc = nn.Linear(m.fc.in_features, n_classes)
            return m
        else:
            raise ValueError(f"Modèle inconnu : {cfg['model_name']}")


# ══════════════════════════════════════════════════════════════════════════════
# MODULE 4 : Trainer
# ══════════════════════════════════════════════════════════════════════════════

class Trainer:
    """
    Gère la boucle complète d'entraînement :
    - Train + Validation à chaque epoch
    - Early Stopping automatique
    - Sauvegarde du meilleur checkpoint
    - Historique complet
    """
    def __init__(self, model, cfg, device):
        self.model  = model.to(device)
        self.cfg    = cfg
        self.device = device
        self.crit   = nn.CrossEntropyLoss()
        self.opt    = Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
        self.sched  = torch.optim.lr_scheduler.CosineAnnealingLR(self.opt, T_max=cfg['epochs'])
        self.history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
        self.best_val_loss  = float('inf')
        self.patience_count = 0

    def _run_epoch(self, loader, training=True):
        self.model.train() if training else self.model.eval()
        total_loss, correct, total = 0.0, 0, 0
        ctx = torch.enable_grad() if training else torch.no_grad()
        with ctx:
            for X, y in loader:
                X, y = X.to(self.device), y.to(self.device)
                if training: self.opt.zero_grad()
                out  = self.model(X)
                loss = self.crit(out, y)
                if training: loss.backward(); self.opt.step()
                total_loss += loss.item() * y.size(0)
                correct    += out.argmax(1).eq(y).sum().item()
                total      += y.size(0)
        return total_loss / total, 100 * correct / total

    def fit(self, train_loader, val_loader):
        print(f"\n🏋️  Entraînement — {self.cfg['epochs']} epochs max\n")
        for epoch in range(self.cfg['epochs']):
            tr_loss, tr_acc = self._run_epoch(train_loader, training=True)
            vl_loss, vl_acc = self._run_epoch(val_loader,   training=False)
            self.sched.step()

            self.history['train_loss'].append(tr_loss)
            self.history['val_loss'].append(vl_loss)
            self.history['train_acc'].append(tr_acc)
            self.history['val_acc'].append(vl_acc)

            # ── Early Stopping & Checkpoint ──────────────────────────────────
            if vl_loss < self.best_val_loss:
                self.best_val_loss = vl_loss
                torch.save(self.model.state_dict(), self.cfg['checkpoint_path'])
                self.patience_count = 0
                marker = ' ✅'
            else:
                self.patience_count += 1
                marker = f' ({self.patience_count}/{self.cfg["early_stop_patience"]})'

            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"  Epoch {epoch+1:>3} | "
                      f"Train {tr_acc:.1f}% / {tr_loss:.4f} | "
                      f"Val {vl_acc:.1f}% / {vl_loss:.4f}{marker}")

            if self.patience_count >= self.cfg['early_stop_patience']:
                print(f"\n⏹️  Early stopping à l'epoch {epoch+1}")
                break

        # Restaurer les meilleurs poids
        self.model.load_state_dict(torch.load(self.cfg['checkpoint_path'], map_location=self.device))
        print(f"✅ Meilleur modèle restauré (val_loss = {self.best_val_loss:.4f})")
        return self.history


pipeline_model   = ModelFactory.build(CFG)
pipeline_trainer = Trainer(pipeline_model, CFG, DEVICE)
pipeline_history = pipeline_trainer.fit(train_loader_p, val_loader_p)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODULE 5 : Evaluator + Module 6 : ModelManager + Module 7 : Predictor
# ══════════════════════════════════════════════════════════════════════════════

class Evaluator:
    """Calcule et affiche les métriques complètes sur le jeu de test."""

    def __init__(self, model, cfg, device):
        self.model, self.cfg, self.device = model, cfg, device

    def evaluate(self, loader):
        self.model.eval()
        all_p, all_t = [], []
        with torch.no_grad():
            for X, y in loader:
                X = X.to(self.device)
                all_p.extend(self.model(X).argmax(1).cpu().numpy())
                all_t.extend(y.numpy())
        all_p, all_t = np.array(all_p), np.array(all_t)
        print("\n📊 Rapport de classification :")
        print(classification_report(all_t, all_p,
              target_names=self.cfg['class_names'], digits=3))
        return all_p, all_t

    def plot_history(self, history):
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        ep = range(1, len(history['train_loss']) + 1)
        axes[0].plot(ep, history['train_loss'], '-',  color='tomato',    lw=2, label='Train')
        axes[0].plot(ep, history['val_loss'],   '--', color='royalblue', lw=2, label='Val')
        axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
        axes[1].plot(ep, history['train_acc'], '-',  color='tomato',    lw=2, label='Train')
        axes[1].plot(ep, history['val_acc'],   '--', color='royalblue', lw=2, label='Val')
        axes[1].set_title('Accuracy (%)'); axes[1].set_xlabel('Epoch'); axes[1].legend()
        plt.suptitle('Pipeline — Courbes d\'entraînement', fontsize=13, fontweight='bold')
        plt.tight_layout(); plt.show()

    def plot_confusion(self, preds, targets):
        cm = confusion_matrix(targets, preds)
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=self.cfg['class_names'],
                    yticklabels=self.cfg['class_names'], ax=ax)
        ax.set_title('Matrice de Confusion — Pipeline Final', fontsize=12)
        ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')
        plt.tight_layout(); plt.show()


class ModelManager:
    """Sauvegarde et chargement du modèle (poids + config)."""

    @staticmethod
    def save(model, cfg, path='final_model.pth'):
        torch.save({'model_state': model.state_dict(), 'config': cfg}, path)
        print(f"💾 Modèle sauvegardé → {path}")

    @staticmethod
    def load(path='final_model.pth', device='cpu'):
        ckpt = torch.load(path, map_location=device)
        model = ModelFactory.build(ckpt['config'])
        model.load_state_dict(ckpt['model_state'])
        model.eval()
        print(f"✅ Modèle chargé depuis {path}")
        return model, ckpt['config']


class Predictor:
    """
    Interface de prédiction pour nouvelles images.
    Accepte : tenseur PyTorch, PIL Image, ou array numpy.
    Retourne : classe prédite + probabilités + visualisation.
    """
    def __init__(self, model, cfg, device):
        self.model, self.cfg, self.device = model, cfg, device
        stats = {'mean': CIFAR_MEAN, 'std': CIFAR_STD}
        self.transform = T.Compose([
            T.Resize((cfg['img_size'], cfg['img_size'])),
            T.ToTensor(),
            T.Normalize(stats['mean'], stats['std'])
        ])

    def predict(self, image, show=True):
        """Prédit la classe d'une image et affiche les résultats."""
        if isinstance(image, np.ndarray):
            image = Image.fromarray(image)
        if isinstance(image, Image.Image):
            tensor = self.transform(image)
        else:
            tensor = image  # Déjà un tenseur

        # Validation de l'entrée
        if tensor.ndim == 3:
            tensor = tensor.unsqueeze(0)
        if tensor.shape[1] != 3:
            raise ValueError(f"Image RGB attendue, reçu {tensor.shape}")

        self.model.eval()
        with torch.no_grad():
            logits = self.model(tensor.to(self.device))
            probs  = torch.softmax(logits, dim=1).cpu().squeeze()
            pred   = probs.argmax().item()

        result = {
            'class_id':   pred,
            'class_name': self.cfg['class_names'][pred],
            'confidence': probs[pred].item(),
            'probabilities': {self.cfg['class_names'][i]: probs[i].item()
                              for i in range(self.cfg['n_classes'])}
        }

        if show:
            self._display(tensor.squeeze(), result)
        return result

    def _display(self, tensor, result):
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        img_display = denorm(tensor).permute(1, 2, 0).numpy()
        axes[0].imshow(np.clip(img_display, 0, 1))
        axes[0].set_title(f"Prédit : {result['class_name']}\n"
                          f"Confiance : {result['confidence']*100:.1f}%", fontsize=12)
        axes[0].axis('off')
        sorted_probs = sorted(result['probabilities'].items(), key=lambda x: -x[1])
        names_p, vals_p = zip(*sorted_probs)
        colors_p = ['royalblue' if n == result['class_name'] else 'steelblue' for n in names_p]
        axes[1].barh(names_p, vals_p, color=colors_p)
        axes[1].set_title('Probabilités par classe', fontsize=11)
        axes[1].set_xlim(0, 1); axes[1].set_xlabel('Probabilité')
        plt.tight_layout(); plt.show()


# ── Évaluation finale ──────────────────────────────────────────────────────
evaluator = Evaluator(pipeline_model, CFG, DEVICE)
evaluator.plot_history(pipeline_history)
preds_final, targets_final = evaluator.evaluate(test_loader_p)
evaluator.plot_confusion(preds_final, targets_final)

# ── Sauvegarde ────────────────────────────────────────────────────────────
ModelManager.save(pipeline_model, CFG, 'final_pipeline_model.pth')

# ── Test du Predictor sur quelques images ─────────────────────────────────
predictor = Predictor(pipeline_model, CFG, DEVICE)
print("\n🎯 Test du Predictor sur 3 images :")
for i in [7, 42, 100]:
    img, lbl = dm.test_ds[i]
    result   = predictor.predict(img, show=True)
    print(f"   Vrai : {CFG['class_names'][lbl]} | Prédit : {result['class_name']} "
          f"({result['confidence']*100:.1f}%)")

---
# 🎓 Conclusion — Récapitulatif des 5 Exercices

| Exercice | Technique principale | Métriques |
|----------|---------------------|----------|
| **1 - Multi-Label** | Sigmoid + BCEWithLogitsLoss | Hamming acc, Exact match, F1 macro |
| **2 - Déséquilibre** | WeightedSampler + Focal Loss | F1 macro, AUC-ROC |
| **3 - Transfer Learning** | EfficientNet-B0, LR différentiel | Accuracy, temps d'entraînement |
| **4 - Interprétabilité** | Saliency, Grad-CAM, LIME | Visualisation qualitative |
| **5 - Pipeline** | DataModule, Trainer, Predictor | Modularité, robustesse |

### 🚀 Pour aller plus loin :
- **Ex1** : Ajouter des couches récurrentes (LSTM) pour capturer les dépendances entre labels
- **Ex2** : Tester ADASYN ou les méthodes de génération (SMOTE) pour l'image
- **Ex3** : Comparer EfficientNet-B0 vs B4 vs ViT (Vision Transformer)
- **Ex4** : Intégrer SHAP pour une explication encore plus précise
- **Ex5** : Déployer le Predictor via une API FastAPI ou une app Gradio